# 02. 전처리와 탐색적 데이터 분석(Preprocessing & EDA)

행동 문항 A1~A10뿐 아니라 나이, 성별, 인종·민족, 황달 이력, 가족 ASD 이력, 거주 국가, 앱 사용 이력, 응답자 관계를 함께 확인합니다.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

raw = pd.read_csv('Autism-Child-Data.csv')
df = raw.copy()
for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].astype('string').str.strip().str.strip("'").str.strip('\"')
    df[c] = df[c].replace('?', np.nan).astype(object)
    df[c] = df[c].where(pd.notna(df[c]), np.nan)
df = df.rename(columns={'jundice':'jaundice','austim':'family_asd','contry_of_res':'country_of_res'})
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df = df.drop_duplicates().reset_index(drop=True)

# 결측값 처리
age_median = df['age'].median()
df['age'] = df['age'].fillna(age_median)
df['ethnicity'] = df['ethnicity'].fillna('Unknown')
df['relation'] = df['relation'].fillna('Unknown')

df['target'] = df['Class/ASD'].map({'NO':0,'YES':1}).astype(int)

BEHAVIOR=[f'A{i}_Score' for i in range(1,11)]
BACKGROUND=['age','gender','ethnicity','jaundice','family_asd','country_of_res','used_app_before','relation']
print('정리 후 크기:',df.shape)

정리 후 크기: (290, 22)


## 1) 결측값과 변수형 확인
- `age` 결측값은 중앙값(Median)으로 대체합니다.
- `ethnicity`, `relation`의 `?`는 특정 범주로 억지로 채우지 않고 `Unknown`으로 유지합니다.
- 범주형 변수는 모델링 단계에서 One-Hot Encoding을 사용합니다.


In [2]:
missing=df[BEHAVIOR+BACKGROUND].isna().sum().sort_values(ascending=False)
display(missing[missing>0].rename('missing').to_frame())
print('수치형:', ['age'] + BEHAVIOR)
print('범주형:', [x for x in BACKGROUND if x!='age'])

,missing
ethnicity,43
relation,43
age,4


수치형: ['age', 'A1_Score', 'A2_Score', 'A3_Score', 'A4_Score', 'A5_Score', 'A6_Score', 'A7_Score', 'A8_Score', 'A9_Score', 'A10_Score']
범주형: ['gender', 'ethnicity', 'jaundice', 'family_asd', 'country_of_res', 'used_app_before', 'relation']


## 2) 행동 문항별 YES/NO 그룹의 1점 비율
이 표는 EDA용이며 최종 변수 선택은 03 노트북의 Selection Train/Validation 절차에서 결정합니다.

In [3]:
rows=[]
for f in BEHAVIOR:
    rows.append({
        'feature':f,
        'overall_rate_1':df[f].mean(),
        'NO_rate_1':df.loc[df.target.eq(0),f].mean(),
        'YES_rate_1':df.loc[df.target.eq(1),f].mean(),
        'gap':df.loc[df.target.eq(1),f].mean()-df.loc[df.target.eq(0),f].mean(),
    })
behavior_rates=pd.DataFrame(rows).sort_values('gap',ascending=False)
display(behavior_rates.round(3))

,feature,overall_rate_1,NO_rate_1,YES_rate_1,gap
3,A4_Score,0.552,0.280,0.843,0.563
8,A9_Score,0.490,0.253,0.743,0.490
7,A8_Score,0.497,0.287,0.721,0.435
9,A10_Score,0.724,0.533,0.929,0.395
0,A1_Score,0.638,0.453,0.836,0.382
5,A6_Score,0.710,0.527,0.907,0.380
2,A3_Score,0.741,0.573,0.921,0.348
4,A5_Score,0.741,0.580,0.914,0.334
6,A7_Score,0.603,0.473,0.743,0.270
1,A2_Score,0.538,0.427,0.657,0.230


## 3) 개인·배경 범주별 Class/ASD YES 비율
표본이 너무 작은 범주는 비율이 크게 흔들릴 수 있으므로 최소 표본 수를 함께 봅니다.

In [4]:
for feature in ['gender','ethnicity','jaundice','family_asd','country_of_res','used_app_before','relation']:
    g=df.groupby(feature,dropna=True)['target'].agg(['count','mean']).reset_index()
    g=g[g['count']>=5].sort_values('mean',ascending=False)
    print('\n---',feature,'---')
    display(g.head(12).rename(columns={'mean':'Class_ASD_YES_rate'}).round(3))


--- gender ---


,gender,count,Class_ASD_YES_rate
1,m,206,0.495
0,f,84,0.452



--- ethnicity ---


,ethnicity,count,Class_ASD_YES_rate
2,Hispanic,7,0.857
1,Black,14,0.643
3,Latino,8,0.625
9,White-European,108,0.574
0,Asian,44,0.477
7,South Asian,21,0.381
5,Others,14,0.357
4,Middle Eastern,27,0.296



--- jaundice ---


...

,country_of_res,count,Class_ASD_YES_rate
51,United States,42,0.738
10,Canada,7,0.714
13,Egypt,9,0.667
3,Australia,23,0.522
50,United Kingdom,49,0.490
18,India,40,0.400
34,New Zealand,13,0.385
6,Bangladesh,6,0.333
24,Jordan,20,0.250
49,United Arab Emirates,7,0.000



--- used_app_before ---


,used_app_before,count,Class_ASD_YES_rate
0,no,279,0.487
1,yes,11,0.364



--- relation ---


,relation,count,Class_ASD_YES_rate
0,Health care professional,13,0.538
1,Parent,212,0.524
2,Relative,17,0.294


### EDA 정리
- 행동 문항과 개인·배경 요인을 모두 후보로 유지합니다.
- 다음 단계에서 18개 요인 각각에 대해 p-value와 연관성 크기를 확인합니다.
- p < 0.05로 관련성이 확인된 요인을 머신러닝 입력 후보로 사용합니다.
- 국가·인종처럼 범주가 많고 표본이 작은 변수는 통계값 해석에 주의합니다.


## 범주가 많은 변수의 추가 정리
모델링 단계에서는 `ethnicity`, `country_of_res`에 대해 Selection Train 기준 10건 미만 범주를 `Other (rare)`로 통합합니다. 이 규칙은 Validation에 그대로 적용합니다.